# Checkpointing and Session State with LangGraph

This tutorial makes session state visible. A deterministic MessagesState graph first prints the full history saved for each thread. It then compares that saved checkpoint history with the smaller message list sent to a model after token-budget trimming.

## Audience and learning goals

Use this after the basic LangGraph ReAct notebook. The offline sections need no credentials or LLM call.

By the end of this notebook, you will be able to:

- continue a conversation with one `thread_id` and prove another thread is isolated;
- see that a fresh `InMemorySaver` loses earlier state;
- distinguish the complete checkpoint from the context passed to a model; and
- keep `SystemMessage` instructions while trimming older context.

## Outline

1. Build a deterministic conversational MessagesState graph.
2. Print saved history after every turn, isolated thread, and fresh saver.
3. Compare checkpointed history with the token-bounded model context.
4. Review the raw-history and rolling-summary contract.
5. Optionally run the same pattern with SAP Generative AI Hub.


## 1. Build a deterministic conversational graph

The assistant below never calls a model. It replies from the MessagesState it receives, which makes the checkpoint behavior reproducible and easy to inspect.


In [1]:
from typing import Any

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.messages.utils import (
    count_tokens_approximately,
    trim_messages,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph


def thread_config(thread_id: str) -> dict[str, dict[str, str]]:
    """Return the LangGraph configuration for one conversation thread."""

    return {"configurable": {"thread_id": thread_id}}


def print_messages(title: str, messages: list[BaseMessage]) -> None:
    """Print every message in a history with its position, role, and content."""

    print(f"\n{title}")
    for position, message in enumerate(messages, start=1):
        print(f"  {position}. {message.type}: {message.content}")


def saved_history(graph: Any, config: dict[str, dict[str, str]]) -> list[BaseMessage]:
    """Read the complete message history currently stored in a graph checkpoint."""

    snapshot = graph.get_state(config)
    return list(snapshot.values["messages"])


def trim_for_model(
    messages: list[BaseMessage],
    max_tokens: int,
) -> list[BaseMessage]:
    """Trim messages with LangChain while preserving the first system instruction."""

    return trim_messages(
        messages,
        max_tokens=max_tokens,
        token_counter=count_tokens_approximately,
        strategy="last",
        start_on="human",
        include_system=True,
        allow_partial=False,
    )


def print_context_comparison(
    source_messages: list[BaseMessage],
    model_messages: list[BaseMessage],
    source_title: str = "Full checkpointed history",
) -> list[BaseMessage]:
    """Print source and sent messages, returning entries excluded from model input."""

    print_messages(source_title, source_messages)
    print_messages("Messages sent to the model after trimming", model_messages)
    omitted_messages = [
        message for message in source_messages if message not in model_messages
    ]
    print("\nOmitted from model input but still available to the application:")
    if omitted_messages:
        for message in omitted_messages:
            print(f"  - {message.type}: {message.content}")
    else:
        print("  - None")
    return omitted_messages


def deterministic_assistant(state: MessagesState) -> dict[str, list[AIMessage]]:
    """Return a predictable reply that shows how many user turns are in state."""

    user_messages = [
        str(message.content)
        for message in state["messages"]
        if isinstance(message, HumanMessage)
    ]
    return {
        "messages": [
            AIMessage(
                content=(
                    f"I can see {len(user_messages)} saved user message(s). "
                    f"Latest: {user_messages[-1]}"
                )
            )
        ]
    }


def build_deterministic_conversation(checkpointer: InMemorySaver):
    """Compile the credential-free MessagesState conversation graph."""

    builder = StateGraph(MessagesState)
    builder.add_node("assistant", deterministic_assistant)
    builder.add_edge(START, "assistant")
    builder.add_edge("assistant", END)
    return builder.compile(checkpointer=checkpointer)


## 2. Continue one thread and inspect every saved turn

The same thread_id is used for two turns. After each invocation, the notebook reads the saved checkpoint directly and prints its whole message list.


In [2]:
checkpointer = InMemorySaver()
conversation = build_deterministic_conversation(checkpointer)
primary_config = thread_config("session-primary")

first_turn = conversation.invoke(
    {
        "messages": [
            SystemMessage(
                content="You are a concise assistant. Preserve the conversation context."
            ),
            HumanMessage(
                content=(
                    "Remember this project briefing: the launch is Thursday, "
                    "the report needs a concise update, and the review is at noon."
                )
            ),
        ]
    },
    primary_config,
)
history_after_first_turn = saved_history(conversation, primary_config)
print_messages("Saved history after first turn", history_after_first_turn)

second_turn = conversation.invoke(
    {
        "messages": [
            HumanMessage(content="What writing style should the report use?")
        ]
    },
    primary_config,
)
history_after_second_turn = saved_history(conversation, primary_config)
print_messages("Saved history after second turn", history_after_second_turn)

assert len(history_after_first_turn) == 3, history_after_first_turn
assert len(history_after_second_turn) == 5, history_after_second_turn
assert isinstance(history_after_second_turn[0], SystemMessage), history_after_second_turn
assert "2 saved user message(s)" in history_after_second_turn[-1].content



Saved history after first turn
  1. system: You are a concise assistant. Preserve the conversation context.
  2. human: Remember this project briefing: the launch is Thursday, the report needs a concise update, and the review is at noon.
  3. ai: I can see 1 saved user message(s). Latest: Remember this project briefing: the launch is Thursday, the report needs a concise update, and the review is at noon.

Saved history after second turn
  1. system: You are a concise assistant. Preserve the conversation context.
  2. human: Remember this project briefing: the launch is Thursday, the report needs a concise update, and the review is at noon.
  3. ai: I can see 1 saved user message(s). Latest: Remember this project briefing: the launch is Thursday, the report needs a concise update, and the review is at noon.
  4. human: What writing style should the report use?
  5. ai: I can see 2 saved user message(s). Latest: What writing style should the report use?


## 3. Prove isolation and fresh-saver loss

A second thread uses the same graph and saver but cannot see the primary history. A fresh saver simulates a restarted process, so even reusing the original thread_id does not restore its old messages.


In [3]:
isolated_config = thread_config("session-isolated")
conversation.invoke(
    {"messages": [HumanMessage(content="This belongs only to the isolated thread.")]},
    isolated_config,
)
isolated_history = saved_history(conversation, isolated_config)
print_messages("Saved history for the isolated thread", isolated_history)

fresh_conversation = build_deterministic_conversation(InMemorySaver())
fresh_turn = fresh_conversation.invoke(
    {"messages": [HumanMessage(content="This is a fresh process session.")]},
    primary_config,
)
fresh_history = saved_history(fresh_conversation, primary_config)
print_messages("Saved history after a fresh saver", fresh_history)

assert "project briefing" not in str(isolated_history)
assert "project briefing" not in str(fresh_history)
assert len(fresh_history) == 2, fresh_history
print("\nCheckpoint assertions passed: continuity, isolation, and fresh-saver loss.")



Saved history for the isolated thread
  1. human: This belongs only to the isolated thread.
  2. ai: I can see 1 saved user message(s). Latest: This belongs only to the isolated thread.

Saved history after a fresh saver
  1. human: This is a fresh process session.
  2. ai: I can see 1 saved user message(s). Latest: This is a fresh process session.

Checkpoint assertions passed: continuity, isolation, and fresh-saver loss.


## 4. Compare checkpointed history with model context

The checkpoint deliberately keeps all five messages from the primary thread. The model receives only the trimmed list below. The omitted messages are explicitly printed and remain available in the checkpoint, so trimming controls model context without deleting session state.


In [4]:
model_context = trim_for_model(history_after_second_turn, max_tokens=72)
omitted_messages = print_context_comparison(
    history_after_second_turn,
    model_context,
)

assert isinstance(model_context[0], SystemMessage), model_context
assert omitted_messages, "The example should omit older messages from model context."
assert all(
    message in history_after_second_turn for message in omitted_messages
), omitted_messages
assert all(message not in model_context for message in omitted_messages), omitted_messages
assert len(history_after_second_turn) == 5, history_after_second_turn
print("\nTrim assertion passed: omitted messages remain checkpointed.")



Full checkpointed history
  1. system: You are a concise assistant. Preserve the conversation context.
  2. human: Remember this project briefing: the launch is Thursday, the report needs a concise update, and the review is at noon.
  3. ai: I can see 1 saved user message(s). Latest: Remember this project briefing: the launch is Thursday, the report needs a concise update, and the review is at noon.
  4. human: What writing style should the report use?
  5. ai: I can see 2 saved user message(s). Latest: What writing style should the report use?

Messages sent to the model after trimming
  1. system: You are a concise assistant. Preserve the conversation context.
  2. human: What writing style should the report use?
  3. ai: I can see 2 saved user message(s). Latest: What writing style should the report use?

Omitted from model input but still available to the application:
  - human: Remember this project briefing: the launch is Thursday, the report needs a concise update, and the revi

## 5. Store raw history and a rolling summary separately

| Data set | Purpose | Lifecycle |
| --- | --- | --- |
| Raw history | Append-only application event or audit record. | Retain, access-control, and delete under the approved business and privacy policy. |
| LangGraph checkpoint | Resume graph execution and thread-scoped state. | Keep only for the required session or workflow lifetime. |
| Rolling summary | Bounded, derived working memory for older facts and decisions. | Regenerate as context changes; never overwrite raw history with it. |

A `thread_id` identifies a checkpoint sequence; it is not authorization. Derive it on the server from the authenticated session, namespace saved data by tenant and user, and authorize each read. Long-term preferences or facts need a separate curated memory store.


## 6. Optional live SAP GenAI Hub ReAct integration

Set RUN_LIVE_DEMO to True only after configuring .env with SAP AI Core credentials and confirming that gpt-5 is deployed. This ReAct example uses `reasoning_effort="none"`: the current Chat Completions path does not support function tools together with active reasoning effort for this model. Use a separate Responses API integration when both capabilities are required. The assistant prints the actual trimmed list passed to the proxy on each turn; the final cell prints the full saved checkpoint and another model-context preview.

The example remains process-local because it uses InMemorySaver. It demonstrates behavior, not durable production persistence.


In [11]:
from dotenv import load_dotenv
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition


LIVE_SYSTEM_MESSAGE = SystemMessage(
    content="Use the multiply tool for multiplication requests and answer concisely."
)


@tool
def multiply(left: float, right: float) -> float:
    """Multiply two numbers for the optional live ReAct demonstration."""

    return left * right


def build_live_react_agent(checkpointer: InMemorySaver):
    """Build a SAP GenAI Hub ReAct agent with visible saved and sent contexts."""

    load_dotenv()
    llm = ChatOpenAI(
        proxy_model_name="gpt-5.6-luna",
        temperature=0.0,
        max_tokens=512,
        reasoning_effort="none",
    )
    llm_with_tools = llm.bind_tools([multiply])

    def assistant(state: MessagesState) -> dict[str, list[BaseMessage]]:
        """Call the proxy with the actual token-bounded context for this turn."""

        full_context = [LIVE_SYSTEM_MESSAGE, *state["messages"]]
        model_context = trim_for_model(full_context, max_tokens=512)
        print_context_comparison(
            full_context,
            model_context,
            source_title="Stable system instruction plus saved checkpoint history",
        )
        response = llm_with_tools.invoke(model_context)
        return {"messages": [response]}

    builder = StateGraph(MessagesState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode([multiply], handle_tool_errors=True))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")
    return builder.compile(checkpointer=checkpointer)


In [16]:
RUN_LIVE_DEMO = True

if RUN_LIVE_DEMO:
    live_agent = build_live_react_agent(InMemorySaver())
    live_config = thread_config("live-session")

    live_agent.invoke(
        {"messages": [HumanMessage(content="What is 6 multiplied by 7?")]},
        live_config,
    )
    live_agent.invoke(
        {
            "messages": [
                HumanMessage(content="What multiplication did I ask you to perform?")
            ]
        },
        live_config,
    )

    live_history = saved_history(live_agent, live_config)
    print_messages("Full saved history for the live session", live_history)
    live_preview = trim_for_model(
        [LIVE_SYSTEM_MESSAGE, *live_history],
        max_tokens=72,
    )
    print_context_comparison(
        [LIVE_SYSTEM_MESSAGE, *live_history],
        live_preview,
        source_title="Stable system instruction plus saved checkpoint history",
    )
else:
    print("Set RUN_LIVE_DEMO to True after configuring .env to call SAP GenAI Hub.")



Stable system instruction plus saved checkpoint history
  1. system: Use the multiply tool for multiplication requests and answer concisely.
  2. human: What is 6 multiplied by 7?

Messages sent to the model after trimming
  1. system: Use the multiply tool for multiplication requests and answer concisely.
  2. human: What is 6 multiplied by 7?

Omitted from model input but still available to the application:
  - None

Stable system instruction plus saved checkpoint history
  1. system: Use the multiply tool for multiplication requests and answer concisely.
  2. human: What is 6 multiplied by 7?
  3. ai: 
  4. tool: 42.0

Messages sent to the model after trimming
  1. system: Use the multiply tool for multiplication requests and answer concisely.
  2. human: What is 6 multiplied by 7?
  3. ai: 
  4. tool: 42.0

Omitted from model input but still available to the application:
  - None

Stable system instruction plus saved checkpoint history
  1. system: Use the multiply tool for multip